# Milestone 4: Baselines across all five horizons

This notebook repeats the Milestone 3 baselines for all five files (`1year` to `5year`). The `1year` file is 5 years before bankruptcy and `5year` is 1 year before, so results are labelled by **years ahead**. We use the training set to fit the models and the validation set to compare them. We are avoiding the test set and will use it later for final evaluation.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 1. Load the cleaned datasets

Each file has 64 financial ratios (inputs) and a `bankrupt` (output) column.
 1 and 0 represent bankrupt and non-bankrupt respectively.

The function below loads the training and validation sets for one file. The table shows how many rows and bankrupt companies each file has.

In [ ]:
YEARS = [1, 2, 3, 4, 5]   # file names; years ahead = 6 - file number

def load_year(year):
    train = pd.read_csv(f'data/cleaned/{year}year/{year}year_train.csv')
    validation = pd.read_csv(f'data/cleaned/{year}year/{year}year_validation.csv')
    return (train.drop(columns='bankrupt'), train['bankrupt'],
            validation.drop(columns='bankrupt'), validation['bankrupt'])

summary = []
for year in YEARS:
    X_train, y_train, X_validation, y_validation = load_year(year)
    summary.append({
        'File': f'{year}year',
        'Years ahead': 6 - year,
        'Training rows': len(X_train),
        'Validation rows': len(X_validation),
        'Bankrupt companies in validation': int(y_validation.sum())
    })
pd.DataFrame(summary).set_index('File')

## 2. The models

We use the same four models as in Milestone 3 for every file, with the same settings:

- The majority benchmark always predicts non-bankruptcy.
- Logistic regression learns from the financial ratios. Its features are scaled using the training set, and the same scaling is then applied to validation data.
- Class-weighted logistic regression gives more weight to the less common class (bankrupt) while training.
- The class-weighted decision tree (maximum depth 5) does not need any type of scaling.

We keep the depth fixed at 5 so that the horizons can be compared fairly. We have not tuned the settings for each file.

In [ ]:
def make_models():
    return {
        'Majority benchmark': DummyClassifier(strategy='most_frequent'),
        'Logistic regression': make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=1000, random_state=42)
        ),
        'Weighted logistic regression': make_pipeline(
            StandardScaler(),
            LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
        ),
        'Weighted tree (depth 5)': DecisionTreeClassifier(
            max_depth=5, class_weight='balanced', random_state=42
        ),
    }

model_order = list(make_models())

## 3. Produce results

Recall measures how many bankrupt companies were correctly identified. 
Precision measures how many companies predicted bankrupt actually were bankrupt. 
F1 combines precision and recall. 

ROC-AUC measures how well the model ranks bankrupt companies above non-bankrupt companies. Average precision summarizes the tradeoff between precision and recall across different cutoffs.

The loop below fits every model on every file and records the scores on the validation set, plus the number of bankrupt companies found and the number of false positives.

In [ ]:
def scores(y_true, predictions, probabilities):
    return {
        'Accuracy': accuracy_score(y_true, predictions),
        'Precision': precision_score(y_true, predictions, zero_division=0),
        'Recall': recall_score(y_true, predictions, zero_division=0),
        'F1': f1_score(y_true, predictions, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, probabilities),
        'Average precision': average_precision_score(y_true, probabilities)
    }

all_results = []
for year in YEARS:
    X_train, y_train, X_validation, y_validation = load_year(year)
    for name, model in make_models().items():
        model.fit(X_train, y_train)
        predictions = model.predict(X_validation)
        probabilities = model.predict_proba(X_validation)[:, 1]
        tn, fp, fn, tp = confusion_matrix(y_validation, predictions, labels=[0, 1]).ravel()
        all_results.append({
            'Years ahead': 6 - year,
            'Model': name,
            **scores(y_validation, predictions, probabilities),
            'Bankrupt found': tp,
            'Bankrupt in validation': int(y_validation.sum()),
            'False positives': fp
        })

results = (pd.DataFrame(all_results)
           .sort_values('Years ahead', ascending=False, kind='stable')
           .set_index(['Years ahead', 'Model']))
results.round(3)

## 4. Check against Milestone 3

The rows for 5 years ahead use the `1year` file, so they should match the Milestone 3 results table (for example, logistic regression recall of 0.024, and the weighted depth-5 tree with recall 0.610 and 162 false positives). If they do not match, something differs from the Milestone 3 notebook.

In [ ]:
results.loc[5].round(3)

## 5. How does performance change with the horizon?

This is the table and chart for our first research question. Each row is a number of years before bankruptcy. The chart reads from left to right as bankruptcy gets closer.

In [ ]:
recall_by_horizon = results['Recall'].unstack('Model')[model_order]
recall_by_horizon.round(3)

In [ ]:
ax = recall_by_horizon.plot(marker='o', figsize=(7, 4))
ax.set_xlabel('Years before bankruptcy')
ax.set_ylabel('Recall (bankrupt class)')
ax.set_xticks([1, 2, 3, 4, 5])
ax.invert_xaxis()
ax.set_ylim(0, 1)
ax.set_title('Validation recall by prediction horizon')
plt.show()

Average precision does not depend on a cutoff, so it shows whether the ranking of companies changes across horizons even when recall at the default cutoff is low.

In [ ]:
average_precision_by_horizon = results['Average precision'].unstack('Model')[model_order]
average_precision_by_horizon.round(3)

## 6. Save the results

The combined table is saved so it can be used in the report and slides.

In [ ]:
os.makedirs('results', exist_ok=True)
results.round(3).to_csv('results/all_horizons_baseline.csv')

### What these results show at a glance

- Compare recall and average precision across the five horizons. Does performance drop as the horizon gets longer?
- Compare the weighted tree with the weighted logistic regression at each horizon, and note the false positives.
- Remember that each validation set has only a small number of bankrupt companies, so small differences are noisy.

These results are preliminary, since they use one fixed split and untuned settings. The test set has not been used.